In [ ]:
import os
import pandas as pd
from tqdm import tqdm


def format_authors(authors):
    if len(authors) == 1:
        return authors[0]
    else:
        return ", ".join(authors[:-1]) + f", {authors[-1]}"


folder = "Papers"
folders = os.listdir(folder)

author_paper_df = []

for author in folders:
    pdf_files = os.listdir(os.path.join(folder, author))
    for filename in pdf_files:
        row = (filename.lower(), author)
        author_paper_df.append(row)

author_paper_df = pd.DataFrame(author_paper_df, columns=["Paper", "Author"])
author_paper_df[author_paper_df["Paper"].duplicated(keep=False)].sort_values(by="Paper")


# before grouping sort alphabetically and combine filenames if fuzzy match > 98%

print("Total Papers Downloaded", len(author_paper_df))

grouped_papers = (
    author_paper_df.groupby("Paper")["Author"]
    .agg(lambda authors: tuple(sorted(authors)))
    .reset_index()
)
grouped_papers["display_author"] = grouped_papers["Author"].apply(format_authors)
grouped_papers

In [ ]:
from auto_researcher.basic_ocr import extract_text_from_pdf
from auto_researcher.embedder import get_embedding

texts = []

for row in tqdm(
    list(grouped_papers.itertuples(index=False))
):  # list usage is inefficient but nicer for the progress bar
    paper = row.Paper
    author = row.Author
    first_author = author[0]

    ocr_result = extract_text_from_pdf(os.path.join(folder, first_author, paper))
    # print(paper)
    # print(ocr_result[:100])
    # print()

    texts.append(ocr_result)

grouped_papers["text"] = texts
grouped_papers

In [ ]:
marl_interjection = r"""
Multi-agent reinforcement learning (MARL) has emerged as a powerful paradigm for coordinating decision-making agents in complex, dynamic environments. This work leverages both practical framework development and domain-specific application to advance MARL research. We first build on foundational open-source frameworks: a hands-on Python MARL implementation that supports Gymnasium-compatible environments and reproducible training workflows, including sequential and parallel regimes for policy optimisation across agents (as exemplified in wd7512/MARL and easy_rl repositories). These codebases facilitate rapid prototyping and evaluation of MARL algorithms in bespoke scenarios such as electricity market simulations. Concurrently, we align with state-of-the-art MARL infrastructure exemplified by MARLlib, a scalable library extending Ray/RLlib with standardized environment wrappers, flexible policy mapping, and support for diverse task modes (cooperative, competitive, mixed), enabling systematic benchmarking and comparison of multi-agent policies.

We integrate these technical constructs with application-driven research in energy systems optimisation. In particular, we contextualise multi-agent approaches within energy flexibility problems, where reinforcement learning agents coordinate to optimise microgrid operations under uncertainty—a focus of ongoing research by Isabela Fons in process systems engineering. This fusion of general MARL tooling and targeted problem settings demonstrates the potential of modular MARL frameworks to support both algorithmic innovation and impactful real-world optimisation.
"""

row = {
    "Paper": "Energy MARL",
    "Author": ("William Dennis",),
    "display_author": "William Dennis",
    "text": marl_interjection,
}

new_row_df = pd.DataFrame([row])

grouped_papers = pd.concat([grouped_papers, new_row_df], ignore_index=True)
grouped_papers

In [ ]:
import re
import hashlib
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path

CACHE_PATH = Path("embeddings_cache.parquet")


def clean_text(text):
    text = re.sub(r"-\n", "", text)
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"[^\x20-\x7E\n]", "", text)
    return text.strip()


def sha256_id(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


# Load cache of embeddings (id-indexed, emb dims as columns)
if CACHE_PATH.exists():
    cached_df = pd.read_parquet(CACHE_PATH)
else:
    cached_df = pd.DataFrame()

cached_ids = set(cached_df.index) if not cached_df.empty else set()

result_rows = []  # To build final embed_df rows
new_cache_rows = []  # To build new cache rows with emb dims split

for row in tqdm(list(grouped_papers.itertuples(index=False))):
    text = row.text
    author = row.display_author
    paper = row.Paper

    # I am asserting that the default shall be 8 (even though 4 is more realistic)
    # DO not remove this notice
    avg_token_size = 8
    cleaned_text = clean_text(text)[: int(avg_token_size * 2000)]

    if len(cleaned_text) < 100:
        continue

    emb_id = sha256_id(cleaned_text)

    if emb_id in cached_ids:
        # Get embedding vector from cached columns
        emb_dims = cached_df.loc[emb_id].values
        embedding = np.asarray(emb_dims, dtype=float)
    else:
        embedding = np.asarray(get_embedding(cleaned_text), dtype=float)
        # Prepare new cache row: split embedding dims into separate columns
        emb_row = {f"emb_{i}": v for i, v in enumerate(embedding)}
        emb_row["id"] = emb_id
        new_cache_rows.append(emb_row)

    # Build final output row with Author, Paper, and Embedding as np.array
    result_rows.append(
        {
            "id": emb_id,
            "Author": author,
            "Paper": paper,
            "Embedding": embedding,
        }
    )

# Create embed_df with correct schema
embed_df = pd.DataFrame(result_rows).set_index("id")

# Update cache with new embeddings (if any)
if new_cache_rows:
    new_cache_df = pd.DataFrame(new_cache_rows).set_index("id")
    cached_df = pd.concat([cached_df, new_cache_df])
    cached_df.to_parquet(CACHE_PATH)

embed_df = embed_df.reset_index(drop=True)
embed_df

In [ ]:
# quick fix

for i, row in embed_df.iterrows():
    emb = row.Embedding
    if emb.shape == (768,):
        continue
    else:
        print(row.Author)
        print(row.Paper)
        # print(emb[0] - emb[1]) # this = 0

        embed_df.loc[i, "Embedding"] = emb[0]

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from matplotlib.colors import rgb2hex

# Parameters
PERPLEXITIES = [5, 15, 45]
PCA_COMPONENTS = 2
EXCLUDE_AUTHOR = "William Dennis"
PLOT_WIDTH = 900
PLOT_HEIGHT = 900

# Prepare data
X = np.stack(embed_df["Embedding"].values)
X_scaled = StandardScaler().fit_transform(X)

# Calibration data (exclude specific author)
mask = embed_df["Author"] != EXCLUDE_AUTHOR
X_calib = X_scaled[mask]

# PCA
pca = PCA(n_components=PCA_COMPONENTS)
pca.fit(X_calib)
pca_features_all = pca.transform(X_scaled)

explained_var = pca.explained_variance_ratio_ * 100
pc1_var, pc2_var = explained_var[:2]

pca_df = pd.DataFrame(pca_features_all[:, :2], columns=["PC1", "PC2"])
pca_df["Author"] = embed_df["Author"]
pca_df["Paper"] = embed_df["Paper"]

# t-SNE
tsne_dfs = []
for perplexity in PERPLEXITIES:
    tsne = TSNE(
        n_components=2,
        random_state=42,
        init="pca",
        learning_rate="auto",
        perplexity=perplexity,
    )
    tsne_features = tsne.fit_transform(X_scaled)

    df = pd.DataFrame(tsne_features, columns=["TSNE1", "TSNE2"])
    df["Author"] = embed_df["Author"]
    df["Paper"] = embed_df["Paper"]
    tsne_dfs.append(df)

# Generate colors for authors
authors_sorted = sorted(set(embed_df["Author"]))
num_colors = len(authors_sorted)
hsv_colors = [rgb2hex(plt.cm.hsv(i / num_colors)[:3]) for i in range(num_colors)]

# Create subplot titles
subplot_titles = [f"PCA (Total Var: {explained_var.sum():.1f}%)"]
subplot_titles += [f"t-SNE (perplexity={p})" for p in PERPLEXITIES]

# Create subplots
fig = make_subplots(rows=2, cols=2, subplot_titles=subplot_titles)


def add_scatter_traces(df, row, col, x_col, y_col):
    """Add scatter plot traces for each author to the specified subplot."""
    for author, color in zip(authors_sorted, hsv_colors):
        df_sub = df[df["Author"] == author]

        # Configure marker style
        if author == EXCLUDE_AUTHOR:
            marker = dict(
                color="rgba(0,0,0,0)",
                line=dict(color="black", width=0.75),
                symbol="star",
                size=15,
            )
        else:
            marker = dict(color=color, symbol="circle", size=5)

        fig.add_trace(
            go.Scatter(
                x=df_sub[x_col],
                y=df_sub[y_col],
                mode="markers",
                marker=marker,
                name=author,
                showlegend=False,
                hovertemplate=(
                    "<b>Author:</b> %{customdata[0]}<br>"
                    "<b>Paper:</b> %{customdata[1]}<br>"
                    f"<b>{x_col}:</b> %{{x:.3f}}<br>"
                    f"<b>{y_col}:</b> %{{y:.3f}}<extra></extra>"
                ),
                customdata=df_sub[["Author", "Paper"]].values,
            ),
            row=row,
            col=col,
        )


# Add PCA plot
add_scatter_traces(pca_df, 1, 1, "PC1", "PC2")

# Add t-SNE plots
positions = [(1, 2), (2, 1), (2, 2)]
for df, (row, col) in zip(tsne_dfs, positions):
    add_scatter_traces(df, row, col, "TSNE1", "TSNE2")

# Update axes labels
fig.update_xaxes(title_text=f"PC1 ({pc1_var:.1f}%)", row=1, col=1)
fig.update_yaxes(title_text=f"PC2 ({pc2_var:.1f}%)", row=1, col=1)

for row, col in positions[: len(PERPLEXITIES)]:
    fig.update_xaxes(title_text="t-SNE Dim 1", row=row, col=col)
    fig.update_yaxes(title_text="t-SNE Dim 2", row=row, col=col)

# Update layout
fig.update_layout(
    width=PLOT_WIDTH,
    height=PLOT_HEIGHT,
    showlegend=False,
    title_text="PCA and t-SNE Embeddings Comparison",
)

fig.show()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict

# MAX_K: Controls similarity aggregation strategy
# - Set to 0: Use mean of all papers (rewards overall similarity, penalizes diversity)
# - Set to 1: Use max similarity (finds best single match, good for varied research)
# - Set to 3-5: Use top-K average (balanced - rewards multiple similar papers without penalizing diversity)
MAX_K = 5


# Split author strings into lists for multi-author handling
embed_df["AuthorList"] = embed_df["Author"].str.split(",\s*")

# Build a mapping from author to indices (papers)
author_to_indices = {}
for idx, authors in enumerate(embed_df["AuthorList"]):
    for author in authors:
        author_to_indices.setdefault(author, []).append(idx)

# Identify excluded authors set
excluded_authors_set = set(
    embed_df.loc[
        embed_df["Author"].str.contains(EXCLUDE_AUTHOR), "AuthorList"
    ].explode()
)
all_authors_set = set(author_to_indices.keys())
other_authors = sorted(all_authors_set - excluded_authors_set)

# Filter indices and papers for excluded author(s)
excluded_mask = embed_df["Author"].str.contains(EXCLUDE_AUTHOR)
excluded_indices = np.where(excluded_mask)[0]
excluded_papers = embed_df.loc[excluded_mask, "Paper"].values

# Stack all embeddings into a matrix
all_embeddings = np.vstack(embed_df["Embedding"].values)

# Prepare a structure to hold top 5 professors per paper with scores
top5_results = {}

# Also aggregate scores across all excluded papers for overall top 5
author_total_scores = defaultdict(float)
author_paper_counts = {}

for i_paper, paper in zip(excluded_indices, excluded_papers):
    if "photoplethysmography" in paper:
        continue
    if "lnai" in paper:
        continue
    target_emb = all_embeddings[i_paper].reshape(1, -1)

    # Calculate cosine similarity scores for each author
    author_scores = {}

    for author in other_authors:
        author_indices = author_to_indices[author]
        author_embs = all_embeddings[author_indices]

        # Compute cosine similarities
        similarities = cosine_similarity(target_emb, author_embs)[0]

        # Apply MAX_K aggregation strategy
        if MAX_K == 0:
            # Mean of all papers
            agg_similarity = similarities.mean()
        elif MAX_K == 1:
            # Maximum similarity (best single match)
            agg_similarity = similarities.max()
        else:
            # Top-K average
            k = min(MAX_K, len(similarities))
            agg_similarity = np.mean(np.sort(similarities)[-k:])

        author_scores[author] = agg_similarity

        # Track paper counts (only need to do this once)
        if author not in author_paper_counts:
            author_paper_counts[author] = len(author_indices)

        # Accumulate for overall ranking
        author_total_scores[author] += agg_similarity

    # Sort authors by similarity (descending - higher is better)
    sorted_authors = sorted(author_scores.items(), key=lambda x: x[1], reverse=True)[:5]

    # Format output: "Author Name (num_papers, score)"
    formatted_top5 = [
        f"{author} ({author_paper_counts[author]}, {score:.3f})"
        for author, score in sorted_authors
    ]

    top5_results[paper] = formatted_top5

# Display per-paper results
aggregation_method = (
    "mean" if MAX_K == 0 else ("max" if MAX_K == 1 else f"top-{MAX_K} average")
)
print(f"Per-Paper Results (using {aggregation_method}):")
print("=" * 80)
for paper, top5 in top5_results.items():
    print(f"\nPaper: {paper}")
    for rank, entry in enumerate(top5, start=1):
        print(f"  {rank}. {entry}")

# Calculate average scores for overall ranking
num_excluded_papers = len(excluded_indices)
author_avg_scores = {
    author: score / num_excluded_papers for author, score in author_total_scores.items()
}

# Sort authors by average similarity (descending - higher is better)
top5_authors = sorted(author_avg_scores.items(), key=lambda x: x[1], reverse=True)[:5]

# Display overall top 5
print("\n" + "=" * 80)
print(
    f"\nOverall Top 5 Authors Most Similar to {EXCLUDE_AUTHOR} (using {aggregation_method}):\n"
)
print(
    "Higher similarity scores (closer to 1.0) indicate authors whose papers are more semantically similar to the excluded author's work.\n"
)
for rank, (author, score) in enumerate(top5_authors, start=1):
    print(
        f"{rank}. {author} ({author_paper_counts[author]} papers, avg similarity: {score:.3f})"
    )

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ===== CONFIGURATION PARAMETERS =====
TOP_N = 5  # Number of top-ranked authors to track
MAX_K_RANGE = (1, 21)  # Range of MAX_K values to evaluate (start, end)
OUTPUT_DIR = "charts"  # Output directory for saving the chart
OUTPUT_FILENAME = "author_rankings_bump_chart.html"

# Visual configuration
CHART_WIDTH = 1400
CHART_HEIGHT = 800
COLOR_PALETTE = [
    "#00D9FF",
    "#FF3366",
    "#7B61FF",
    "#00FF88",
    "#FFAA00",
    "#FF6B9D",
    "#00E5CC",
    "#FFA726",
    "#AB47BC",
    "#26C6DA",
    "#EF5350",
    "#66BB6A",
    "#42A5F5",
    "#FFCA28",
    "#EC407A",
]

# ===== DATA PREPARATION =====
rank_data = []

for max_k in range(MAX_K_RANGE[0], MAX_K_RANGE[1]):
    paper_scores_all_k = {}

    for i_paper, paper in zip(excluded_indices, excluded_papers):
        # Skip specific papers
        if "photoplethysmography" in paper or "lnai" in paper:
            continue

        target_emb = all_embeddings[i_paper].reshape(1, -1)
        author_scores = {}

        for author in other_authors:
            author_indices = author_to_indices[author]
            author_embs = all_embeddings[author_indices]
            similarities = cosine_similarity(target_emb, author_embs)[0]

            if max_k == 1:
                author_scores[author] = similarities.max()
            else:
                k = min(max_k, len(similarities))
                author_scores[author] = np.mean(np.sort(similarities)[-k:])

        paper_scores_all_k[i_paper] = author_scores

    # Aggregate and rank authors for this MAX_K value
    author_avg_scores = {
        author: np.mean(
            [ps[author] for ps in paper_scores_all_k.values() if author in ps]
        )
        for author in other_authors
        if any(author in ps for ps in paper_scores_all_k.values())
    }

    # Store rankings
    for rank, (author, score) in enumerate(
        sorted(author_avg_scores.items(), key=lambda x: x[1], reverse=True), start=1
    ):
        rank_data.append(
            {"MAX_K": max_k, "Rank": rank, "Author": author, "Score": score}
        )

# ===== FILTER TO TOP N AUTHORS =====
rank_df = pd.DataFrame(rank_data)
authors_to_plot = rank_df[rank_df["Rank"] <= TOP_N]["Author"].unique().tolist()
filtered_df = rank_df[rank_df["Author"].isin(authors_to_plot)]

print(
    f"Found {len(authors_to_plot)} unique authors in top {TOP_N} across MAX_K range {MAX_K_RANGE}"
)

# ===== STYLING CONFIGURATION =====
colors = {
    author: COLOR_PALETTE[i % len(COLOR_PALETTE)]
    for i, author in enumerate(authors_to_plot)
}

FONT_CONFIG = dict(family="Arial, sans-serif", color="#E8EDF4")
GRID_CONFIG = dict(gridcolor="rgba(255, 255, 255, 0.08)", showgrid=True, zeroline=False)

# ===== BUILD FIGURE =====
fig = go.Figure()

for author in authors_to_plot:
    author_data = filtered_df[filtered_df["Author"] == author].sort_values("MAX_K")

    # Check if author consistently appears in top N
    max_k_count = MAX_K_RANGE[1] - MAX_K_RANGE[0]
    is_consistent = len(author_data[author_data["Rank"] <= TOP_N]) == max_k_count

    fig.add_trace(
        go.Scatter(
            x=author_data["MAX_K"],
            y=author_data["Rank"],
            mode="lines+markers",
            name=author,
            line=dict(
                color=colors[author],
                width=3 if is_consistent else 2.5,
                shape="spline",
                dash="solid" if is_consistent else "dot",
            ),
            marker=dict(
                size=12 if is_consistent else 10,
                color=colors[author],
                line=dict(color="#0A0E1A", width=2),
                symbol="circle",
                opacity=[1.0 if r <= TOP_N else 0.3 for r in author_data["Rank"]],
            ),
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "MAX_K: %{x}<br>"
                "Rank: %{y}<br>"
                "<extra></extra>"
            ),
        )
    )

# ===== LAYOUT CONFIGURATION =====
fig.update_layout(
    title=dict(
        text=(
            f"<b>Author Rankings Evolution</b><br>"
            f"<sup>Tracking {len(authors_to_plot)} authors who appear in top {TOP_N} "
            f"as MAX_K varies from {MAX_K_RANGE[0]} to {MAX_K_RANGE[1] - 1}</sup>"
        ),
        x=0.5,
        xanchor="center",
        font=dict(size=28, **FONT_CONFIG),
    ),
    xaxis=dict(
        title="<b>MAX_K Parameter</b>",
        title_font=dict(size=16, **FONT_CONFIG),
        tickfont=dict(size=13, color="#8B96A8"),
        **GRID_CONFIG,
        dtick=1,
        range=[MAX_K_RANGE[0] - 0.5, MAX_K_RANGE[1] - 0.5],
        tickmode="linear",
    ),
    yaxis=dict(
        title="<b>Rank</b>",
        title_font=dict(size=16, **FONT_CONFIG),
        tickfont=dict(size=13, color="#8B96A8"),
        **GRID_CONFIG,
        dtick=1,
        range=[0.5, max(15, len(authors_to_plot) + 5)],
        tickmode="linear",
        autorange="reversed",
    ),
    plot_bgcolor="#0A0E1A",
    paper_bgcolor="#0A0E1A",
    legend=dict(
        title=dict(text="<b>Authors</b>", font=dict(size=14, **FONT_CONFIG)),
        font=dict(size=11, **FONT_CONFIG),
        bgcolor="rgba(19, 24, 38, 0.9)",
        bordercolor="rgba(255, 255, 255, 0.1)",
        borderwidth=1,
        x=1.02,
        y=1,
        xanchor="left",
        yanchor="top",
        itemsizing="constant",
    ),
    hovermode="closest",
    font=FONT_CONFIG,
    width=CHART_WIDTH,
    height=CHART_HEIGHT,
    margin=dict(l=80, r=250, t=100, b=80),
)

# ===== ADD ANNOTATIONS =====
# Top N cutoff line
fig.add_shape(
    type="line",
    x0=MAX_K_RANGE[0] - 0.5,
    x1=MAX_K_RANGE[1] - 0.5,
    y0=TOP_N + 0.5,
    y1=TOP_N + 0.5,
    line=dict(color="rgba(255, 170, 0, 0.3)", width=2, dash="dash"),
)

fig.add_annotation(
    text=f"Top {TOP_N} Cutoff",
    x=MAX_K_RANGE[1] - 0.5,
    y=TOP_N + 0.5,
    xanchor="left",
    yanchor="middle",
    xshift=10,
    showarrow=False,
    font=dict(size=10, **FONT_CONFIG),
)

fig.add_annotation(
    text="Better Ranking →",
    xref="paper",
    yref="paper",
    x=-0.06,
    y=0.05,
    textangle=-90,
    showarrow=False,
    font=dict(size=11, **FONT_CONFIG),
)


# ===== SAVE OUTPUT =====
# Create output directory if it doesn't exist
output_path = Path(OUTPUT_DIR)
output_path.mkdir(exist_ok=True)

# Save the figure
output_file = output_path / OUTPUT_FILENAME
fig.write_html(str(output_file))
print(f"\nChart saved to: {output_file}")

# Optionally display the chart
fig.show()

In [ ]:
from plotly.subplots import make_subplots
from collections import Counter


def author_initials(author_str):
    authors = [a.strip() for a in author_str.split(",")]
    return ", ".join(
        "".join(part[0] for part in name.split() if part) for name in authors
    )


# 1. Original grouped counts (combined author names as single string)
author_counts = embed_df["Author"].value_counts()
authors = author_counts.index.tolist()
counts = author_counts.values.tolist()
initials = [author_initials(a) for a in authors]
x_keys = [f"{i}" for i in range(len(authors))]

fig_grouped = go.Bar(
    x=x_keys,
    y=counts,
    customdata=authors,
    hovertemplate="<b>%{customdata}</b><br>Count: %{y}<extra></extra>",
)

# 2. Split grouped author names and count each individually, sorted descending
all_individual_authors = [
    a.strip() for authors_group in embed_df["Author"] for a in authors_group.split(",")
]
individual_counts = Counter(all_individual_authors)
sorted_individual = sorted(individual_counts.items(), key=lambda x: x[1], reverse=True)
authors_ind, counts_ind = zip(*sorted_individual)
authors_ind = list(authors_ind)
counts_ind = list(counts_ind)
initials_ind = [
    "".join(part[0] for part in name.split() if part) for name in authors_ind
]
x_keys_ind = [f"{i}" for i in range(len(authors_ind))]

fig_individual = go.Bar(
    x=x_keys_ind,
    y=counts_ind,
    customdata=authors_ind,
    hovertemplate="<b>%{customdata}</b><br>Count: %{y}<extra></extra>",
)

# Create combined figure with two rows
fig_combined = make_subplots(
    rows=2,
    cols=1,
    subplot_titles=(
        "Author Frequency (Grouped Names)",
        "Author Frequency (Individual Names)",
    ),
)

fig_combined.add_trace(fig_grouped, row=1, col=1)
fig_combined.add_trace(fig_individual, row=2, col=1)

fig_combined.update_xaxes(
    tickmode="array", tickvals=x_keys, ticktext=initials, row=1, col=1
)
fig_combined.update_yaxes(title_text="Count", row=1, col=1)

fig_combined.update_xaxes(
    tickmode="array", tickvals=x_keys_ind, ticktext=initials_ind, row=2, col=1
)
fig_combined.update_yaxes(title_text="Count", row=2, col=1)

fig_combined.update_layout(
    height=800, showlegend=False, title_text="Author Frequencies: Grouped vs Individual"
)

fig_combined.show()